# LLM Classification — Single Notebook Submission

Kaggle Code Competition: predict human preference (A/B/Tie).

**Pipeline:** TF-IDF → SBERT → DeBERTa-small → Ensemble
**Hardware:** T4/P100 GPU (auto-detected), TPU fallback
**Target:** log_loss < 0.85

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
  for filename in filenames:
      print(os.path.join(dirname, filename))

In [ ]:
import os
import gc
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from scipy.sparse import hstack

import torch
from pathlib import Path

# Detect device: TPU > GPU > CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
is_tpu = False

# TPU detection (if torch_xla available)
try:
    import torch_xla.core.xla_model as xm
    test_device = xm.xla_device()
    if test_device.type == 'xla':
        DEVICE = test_device
        is_tpu = True
        print(f'TPU v5e-8 detected: {DEVICE}')
    else:
        print(f'GPU detected: {DEVICE}')
except Exception:
    print(f'GPU/CPU detected: {DEVICE}')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DATA_DIR = Path('/kaggle/input/competitions/llm-classification-finetuning')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Training Device: {DEVICE}')

## Load Data

In [ ]:
# Load train & test
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv') if (DATA_DIR / 'sample_submission.csv').exists() else None

print(f'Train: {len(train_df)}, Test: {len(test_df)}')

# Target: 0=A, 1=B, 2=Tie
train_df['target'] = (train_df['winner_model_a'].astype(int) * 0 +
                      train_df['winner_model_b'].astype(int) * 1 +
                      train_df['winner_tie'].astype(int) * 2)

print(f'Target distribution:\n{train_df["target"].value_counts().sort_index()}')

## Phase 1: TF-IDF + LogReg

In [ ]:
print('\n=== Phase 1: TF-IDF ===')

# Text features
train_text = (train_df['prompt'].fillna('') + ' ' +
              train_df['response_a'].fillna('') + ' ' +
              train_df['response_b'].fillna(''))
test_text = (test_df['prompt'].fillna('') + ' ' +
             test_df['response_a'].fillna('') + ' ' +
             test_df['response_b'].fillna(''))

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(train_text)
X_test_tfidf = vectorizer.transform(test_text)

# Handcrafted features
def handcrafted_features(df):
    feats = pd.DataFrame()
    feats['len_a'] = df['response_a'].str.len()
    feats['len_b'] = df['response_b'].str.len()
    feats['len_ratio'] = feats['len_a'] / (feats['len_b'] + 1)
    feats['word_count_a'] = df['response_a'].str.split().str.len()
    feats['word_count_b'] = df['response_b'].str.split().str.len()
    feats['word_ratio'] = feats['word_count_a'] / (feats['word_count_b'] + 1)
    feats['prompt_len'] = df['prompt'].str.len()
    feats['total_len'] = feats['len_a'] + feats['len_b']
    return feats.fillna(0).values

X_train_hand = handcrafted_features(train_df)
X_test_hand = handcrafted_features(test_df)

# Combine → CSR for fast row slicing
X_train_tfidf_full = hstack([X_train_tfidf, X_train_hand]).tocsr()
X_test_tfidf_full = hstack([X_test_tfidf, X_test_hand]).tocsr()
y_train = train_df['target'].values

print(f'TF-IDF features: {X_train_tfidf_full.shape[1]}')

# 5-fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
oof_tfidf = np.zeros((X_train_tfidf_full.shape[0], 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_tfidf_full, y_train)):
    model = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs')
    model.fit(X_train_tfidf_full[train_idx], y_train[train_idx])
    oof_tfidf[val_idx] = model.predict_proba(X_train_tfidf_full[val_idx])
    print(f'  Fold {fold+1}: log_loss={log_loss(y_train[val_idx], oof_tfidf[val_idx]):.4f}')

print(f'TF-IDF OOF log_loss: {log_loss(y_train, oof_tfidf):.4f}')

# Final TF-IDF model
model_tfidf = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs')
model_tfidf.fit(X_train_tfidf_full, y_train)
test_preds_tfidf = model_tfidf.predict_proba(X_test_tfidf_full)

# Free memory
del X_train_tfidf, X_test_tfidf, X_train_tfidf_full, X_test_tfidf_full
gc.collect()

## Phase 2: SBERT Embeddings + LogReg + TabPFN

In [ ]:
print('\n=== Phase 2: SBERT ===')
from sentence_transformers import SentenceTransformer

model_name = 'all-MiniLM-L6-v2'
print(f'Loading {model_name}...')

# SBERT: GPU if available
SBERT_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
sbert = SentenceTransformer(model_name, device=SBERT_DEVICE)
print(f'SBERT device: {SBERT_DEVICE}')

def encode_pairs(texts_a, texts_b, prompts, batch_size=64):
    # Encode all three separately
    emb_a = sbert.encode(texts_a.fillna(''), show_progress_bar=True, batch_size=batch_size)
    emb_b = sbert.encode(texts_b.fillna(''), show_progress_bar=True, batch_size=batch_size)
    emb_prompt = sbert.encode(prompts.fillna(''), show_progress_bar=True, batch_size=batch_size)

    norm_a = emb_a / np.linalg.norm(emb_a, axis=1, keepdims=True)
    norm_b = emb_b / np.linalg.norm(emb_b, axis=1, keepdims=True)
    norm_prompt = emb_prompt / np.linalg.norm(emb_prompt, axis=1, keepdims=True)

    cos_sim = np.sum(norm_a * norm_b, axis=1)
    diff = emb_a - emb_b

    len_a = np.array([len(str(s)) for s in texts_a])
    len_b = np.array([len(str(s)) for s in texts_b])
    len_ratio = len_a / (len_b + 1)
    word_count_a = np.array([len(str(s).split()) for s in texts_a])
    word_count_b = np.array([len(str(s).split()) for s in texts_b])
    word_ratio = word_count_a / (word_count_b + 1)
    hand = np.column_stack([len_a, len_b, len_ratio, word_count_a, word_count_b, word_ratio])

    # emb_a, emb_b, emb_prompt added (~1158 total dims vs ~390 before)
    features = np.concatenate([emb_a, emb_b, emb_prompt, diff, cos_sim[:, None], hand], axis=1)
    return features

print('Encoding train...')
X_train_sbert = encode_pairs(train_df['response_a'], train_df['response_b'], train_df['prompt'])
print('Encoding test...')
X_test_sbert = encode_pairs(test_df['response_a'], test_df['response_b'], test_df['prompt'])

print(f'SBERT features: {X_train_sbert.shape[1]}')

In [ ]:
# SBERT + LogReg
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
oof_sbert = np.zeros((len(X_train_sbert), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_sbert, y_train)):
    model = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs')
    model.fit(X_train_sbert[train_idx], y_train[train_idx])
    oof_sbert[val_idx] = model.predict_proba(X_train_sbert[val_idx])
    print(f'  Fold {fold+1}: log_loss={log_loss(y_train[val_idx], oof_sbert[val_idx]):.4f}')

print(f'SBERT OOF log_loss: {log_loss(y_train, oof_sbert):.4f}')

# Final SBERT model
model_sbert = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs')
model_sbert.fit(X_train_sbert, y_train)
test_preds_sbert = model_sbert.predict_proba(X_test_sbert)

In [ ]:
# TabPFN on SBERT embeddings (if within limits)
n_samples, n_features = X_train_sbert.shape
test_preds_tabpfn = None
oof_tabpfn = None

if n_samples <= 10000 and n_features <= 2000:
    print('\n=== TabPFN ===')
    try:
        from tabpfn import TabPFNClassifier
        from sklearn.model_selection import cross_val_predict
        
        # TabPFN on CPU — GPU has CUDA compat issues on Kaggle
        clf_tabpfn = TabPFNClassifier(n_estimators=8, device='cpu', random_state=RANDOM_SEED)
        oof_tabpfn = cross_val_predict(clf_tabpfn, X_train_sbert, y_train, cv=5, method='predict_proba')
        print(f'TabPFN OOF log_loss: {log_loss(y_train, oof_tabpfn):.4f}')
        
        clf_tabpfn.fit(X_train_sbert, y_train)
        test_preds_tabpfn = clf_tabpfn.predict_proba(X_test_sbert)
    except Exception as e:
        print(f'TabPFN skipped: {e}')
else:
    print(f'TabPFN skipped: {n_samples} samples, {n_features} features (limit: 10000, 2000)')

## Phase 3: DeBERTa Small Finetuning (TPU)

In [ ]:
# Model config
MODEL_NAME = 'microsoft/deberta-v3-small'
MAX_LENGTH = 512  # was 384
BATCH_SIZE = 16
GRAD_ACCUM = 2
EPOCHS = 3
HEAD_LR_MULT = 0.3
WARMUP_RATIO = 0.1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Device: {DEVICE}, Model: {MODEL_NAME}, Batch: {BATCH_SIZE}')

In [ ]:
# ComparisonDataset (for DeBERTa)
class ComparisonDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, is_train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row["prompt"])
        resp_a = str(row["response_a"])
        resp_b = str(row["response_b"])

        # Position randomization during training (50% flip)
        if self.is_train and np.random.rand() < 0.5:
            resp_a, resp_b = resp_b, resp_a
            target = row["target"]
            if target == 0:
                target = 1
            elif target == 1:
                target = 0
        else:
            if "target" in row:
                target = row["target"]
            else:
                target = 0  # dummy for test

        encoding = self.tokenizer(
            prompt,
            resp_a + " [SEP] " + resp_b,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(target, dtype=torch.long)
        }

# Split train/val (last 10% for validation)
val_size = int(0.1 * len(train_df))
train_subset = train_df.iloc[:-val_size].reset_index(drop=True)
val_subset = train_df.iloc[-val_size:].reset_index(drop=True)

train_dataset = ComparisonDataset(train_subset, tokenizer, MAX_LENGTH, is_train=True)
val_dataset = ComparisonDataset(val_subset, tokenizer, MAX_LENGTH, is_train=False)
test_dataset = ComparisonDataset(test_df, tokenizer, MAX_LENGTH, is_train=False)

In [ ]:
# DataLoader: pin_memory only for GPU
pin_mem = DEVICE.type == 'cuda'
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=pin_mem)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0, pin_memory=pin_mem)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0, pin_memory=pin_mem)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

In [ ]:
# Load model and move to device
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model = model.to(DEVICE)

# Differential LR: lower for classifier head (newly initialized)
head_params = [p for n, p in model.named_parameters() if 'classifier' in n or 'pooler' in n]
backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n and 'pooler' not in n]

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1e-6},
    {'params': head_params, 'lr': 3e-7},
], weight_decay=0.01, eps=1e-6)

total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f'Training: {EPOCHS} epochs, batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM}')
print(f'Total steps: {total_steps}, warmup steps: {warmup_steps}')

In [ ]:
# Training loop — pure FP32 with BTD auxiliary loss
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    btd_loss = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for step, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        ce_loss = outputs.loss / GRAD_ACCUM

        # BTD auxiliary loss (weight 0.1)
        # logits: [logit_A_wins, logit_B_wins, logit_tie]
        logits = outputs.logits
        eta = logits[:, 0] - logits[:, 1]   # A-B advantage
        nu = logits[:, 2]                     # tie logit

        labels_float = labels.float()
        btd_ce = -(
            labels_float * torch.log(torch.sigmoid(eta) + 1e-7) +
            (1 - labels_float) * torch.log(1 - torch.sigmoid(eta) + 1e-7)
        )
        tie_loss = -(labels_float == 2).float() * torch.log(torch.sigmoid(nu) + 1e-7) - (labels_float != 2).float() * torch.log(1 - torch.sigmoid(nu) + 1e-7)

        btd = (btd_ce + tie_loss).mean()
        loss = ce_loss + 0.1 * btd
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        epoch_loss += ce_loss.item() * GRAD_ACCUM
        btd_loss += btd.item()
        pbar.set_postfix({'loss': f'{epoch_loss/(step+1):.4f}', 'btd': f'{btd_loss/(step+1):.4f}'})

    print(f'Epoch {epoch+1}: avg_loss={epoch_loss/len(train_loader):.4f}, avg_btd={btd_loss/len(train_loader):.4f}')

In [ ]:
# Validation predictions
print('Validating...')
model.eval()
val_preds = []
with torch.no_grad():
    for batch in tqdm(val_loader, desc='Validation'):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        val_preds.append(probs)

val_preds = np.vstack(val_preds)
val_loss = log_loss(val_subset['target'].values, val_preds)
print(f'DeBERTa val log_loss: {val_loss:.4f}')

# OOF (partial - only last 10%)
oof_deberta = np.zeros((len(train_df), 3))
oof_deberta[-val_size:] = val_preds

In [ ]:
# Test predictions
print('Generating test predictions...')
model.eval()
test_preds_deberta = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test'):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        test_preds_deberta.append(probs)

test_preds_deberta = np.vstack(test_preds_deberta)
print(f'Test predictions shape: {test_preds_deberta.shape}')

# Clean up
del model
gc.collect()
torch.cuda.empty_cache()

## Phase 4: Ensemble

In [ ]:
print('\n=== Phase 4: Ensemble ===')

y_true = train_df['target'].values

# Collect all predictions
names = ['tfidf', 'sbert', 'deberta']
oof_dict = {'tfidf': oof_tfidf, 'sbert': oof_sbert, 'deberta': oof_deberta}
test_dict = {'tfidf': test_preds_tfidf, 'sbert': test_preds_sbert, 'deberta': test_preds_deberta}

if test_preds_tabpfn is not None:
    names.append('tabpfn')
    oof_dict['tabpfn'] = oof_tabpfn
    test_dict['tabpfn'] = test_preds_tabpfn

print(f'Models: {names}')

# Normalize OOF predictions per row before ensembling
for name in names:
    oof_dict[name] = oof_dict[name] / oof_dict[name].sum(axis=1, keepdims=True)

# Grid search weights — dynamic for N models, step 0.1
from itertools import product

n_models = len(names)
best_loss = 999
best_weights = None

# Build weight grid iteratively — step 0.1 (was 0.2)
weight_ranges = [np.arange(0.0, 1.05, 0.1) for _ in range(n_models - 1)]
for weights in product(*weight_ranges):
    weights = list(weights)
    remaining = 1.0 - sum(weights)
    if remaining < -1e-6 or remaining > 1.0:
        continue
    weights.append(remaining)

    ens_oof = sum(w * oof_dict[names[i]] for i, w in enumerate(weights))
    loss = log_loss(y_true, ens_oof)
    if loss < best_loss:
        best_loss = loss
        best_weights = {names[i]: w for i, w in enumerate(weights)}

print(f'Best OOF log_loss: {best_loss:.4f}')
print(f'Best weights: {best_weights}')

In [ ]:
# Normalize test predictions per row before weighting
for name in names:
    test_dict[name] = test_dict[name] / test_dict[name].sum(axis=1, keepdims=True)

# Final predictions
final_preds = np.zeros_like(test_preds_tfidf)
for name, w in best_weights.items():
    if w > 0:
        final_preds += w * test_dict[name]

# Final normalize
final_preds = final_preds / final_preds.sum(axis=1, keepdims=True)

print(f'Final predictions shape: {final_preds.shape}')
print(f'Row sums: min={final_preds.sum(axis=1).min():.6f}, max={final_preds.sum(axis=1).max():.6f}')

## Generate Submission

In [ ]:
# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': final_preds[:, 0],
    'winner_model_b': final_preds[:, 1],
    'winner_tie': final_preds[:, 2],
})

# Save
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f'Saved to {OUTPUT_DIR / "submission.csv"}')
print(submission.head())

# Verify
row_sums = submission[['winner_model_a', 'winner_model_b', 'winner_tie']].sum(axis=1)
print(f'\nVerification: min={row_sums.min():.6f}, max={row_sums.max():.6f}')

assert (final_preds >= 0).all() and (final_preds <= 1).all(), 'Predictions out of range!'
print('All checks passed!')